<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Camera Calibration</b></h1>
</div>

## Context

Camera calibration establishes the geometric relationship between known points on a calibration target and their measured image coordinates. This laboratory calibrates a pinhole camera from multiple views of one planar chessboard and evaluates the resulting geometry in pixel space.

## Problem Statement

Given several images of the same planar chessboard, with known square size and corresponding detected image corners, estimate:

- the shared intrinsic camera matrix $K$;
- one rotation matrix $R$ for each valid image;
- one translation vector $t$ for each valid image.

The solution must use **normalized DLT** to estimate one planar homography $H$ per valid view and **Zhang's planar calibration method** to recover $K$. The calibrated model is then validated by reprojecting the known planar points into each image and measuring the pixel-space residuals.

Lens-distortion coefficients and nonlinear bundle-adjustment refinement are outside the implemented scope.

## Inputs / Provided Data

| Item | Implementation value |
| --- | ---: |
| Image location | `../data/calibration_images/*.jpg` |
| Image ordering | Sorted by filename |
| Internal chessboard corners | $8 \times 6$ |
| Square size | $0.03\,\mathrm{m}$ |
| Minimum valid views | 3 |
| Target | Planar chessboard, $Z=0$ |
| Corner refinement | OpenCV sub-pixel refinement |
| Geometry unit | metres |
| Reprojection-error unit | pixels |

## Objectives

1. Discover and validate the calibration images.
2. Detect the $8 \times 6$ internal chessboard corners.
3. Refine detected corners to sub-pixel precision.
4. Generate planar coordinates with the first corner at the origin, $X$ along the 8-corner direction and $Y$ along the 6-corner direction.
5. Normalize image and planar point sets.
6. Estimate one normalized-DLT homography $H$ for every valid view.
7. Stack Zhang constraints and solve $Vb=0$ by SVD.
8. Recover $K$ from $b$.
9. Recover one pose $(R,t)$ per valid view.
10. Reproject all calibration points and compute pixel-space residuals.
11. Report per-view and global reprojection metrics.
12. Save the diagnostic figures and validate the final outputs.

## Constraints and Assumptions

- The calibration target is planar and its geometry is known.
- A valid view must contain a detectable complete $8 \times 6$ internal-corner pattern.
- At least three valid views must remain after detection.
- Point normalization follows the centroid / mean-distance-$\sqrt{2}$ convention.
- Homographies are estimated by normalized DLT and normalized so that $H_{33}=1$.
- Camera intrinsics are recovered from Zhang's closed-form constraints.
- The recovered rotation is projected to the nearest proper rotation matrix by SVD and constrained to $\det(R)=+1$.
- Radial and tangential lens distortion are not estimated.
- All data and output paths remain repository-relative.

## Required Tasks

1. Load the sorted JPEG calibration images.
2. Detect and refine chessboard corners.
3. Build the planar world coordinates.
4. Compute $T_{\mathrm{image}}$ and $T_{\mathrm{plane}}$.
5. Build the DLT matrix $Q$ and solve $Q\mathbf{h}=0$ by SVD.
6. Denormalize each homography with $H=T_{\mathrm{image}}^{-1}H_nT_{\mathrm{plane}}$.
7. Build the Zhang matrix $V$ and solve $Vb=0$ by SVD.
8. Recover $\alpha,\beta,\gamma,u_0,v_0$ and construct $K$.
9. Recover $R$ and $t$ for every retained view.
10. Reproject the $Z=0$ calibration points.
11. Compute point-wise Euclidean errors, mean error and RMSE.
12. Produce and save the six required diagnostic figures.
13. Run the numerical and output-file validation checks.

## Expected Outputs

### Numerical results

- intrinsic matrix $K$;
- intrinsic parameters $\alpha,\beta,\gamma,u_0,v_0$;
- Zhang constraint singular values and residual $\lVert Vb\rVert$;
- per-view rotation matrix $R$ and translation vector $t$;
- per-view mean reprojection error and RMSE;
- overall mean reprojection error and RMSE.

### Saved figures

- `detected_chessboard_corners.png`
- `homography_estimation_pipeline.png`
- `estimated_camera_poses.png`
- `reprojection_results.png`
- `mean_reprojection_error_by_view.png`
- `reprojection_error_distribution.png`

## Success Criteria

The implemented solution is considered complete when:

- at least three calibration views are valid;
- every retained view contains exactly $8 \times 6=48$ image and planar points;
- every retained homography contains finite values;
- $K$ is a finite $3 \times 3$ matrix;
- each recovered $R$ satisfies $R^TR\approx I$ and $\det(R)\approx1$;
- every reprojection residual is finite;
- all six required figures exist in `../outputs/figures/`;
- the implementation executes from top to bottom using repository-relative paths.

## Notebook Responsibilities

- `camera_calibration_problem_statement.ipynb` — defines exactly what is solved and how success is judged.
- `camera_calibration_theory.ipynb` — derives the mathematics implemented in code.
- `camera_calibration_requirements_gathering_and_approach.ipynb` — maps requirements to the concrete engineering pipeline.
- `camera_calibration.ipynb` — executable source of truth for the implemented solution.